# 통번역과 AI III — Midterm Workshop
한국외국어대학교 통번역대학원

- 성함: (이 텍스트를 더블클릭하여 본인 이름을 적어주세요)
- 학번: (이 텍스트를 더블클릭하여 본인 학번을 적어주세요)

---

### 오늘의 목표

ChatGPT나 Gemini에 텍스트를 붙여넣고 "번역해줘"라고 하면 한두 문장은 쉽게 됩니다.
하지만 **책 한 권 분량의 이미지**에서 텍스트를 뽑아 번역하려면? 한 번에 되지 않습니다.

오늘은 이런 **대규모 번역 작업을 자동화하는 파이프라인**을 직접 구축해봅니다:

1. **바이브 코딩** — AI 챗봇에게 코드를 요청하여 작업을 자동화하는 방법을 체험한다
2. **OCR** — 이미지 속 텍스트를 AI로 추출한다
3. **LLM 번역 + 프롬프트 엔지니어링** — 번역 품질을 프롬프트로 제어하는 방법을 익힌다
4. **자동 평가** — BLEU와 COMETKiwi로 번역 품질을 측정하고 비교한다

---

> 코드를 직접 작성할 필요는 없습니다. **AI 챗봇(ChatGPT, Gemini, Claude 등)에게 코드를 요청** 하거나, **제공된 코드를 실행** 하면서 진행합니다.

## 0. 환경 설정

### 0.1 구글 드라이브 마운트

아래 셀을 실행하기 전에, 이미지 폴더를 내 드라이브에 추가해야 합니다.

1. [이미지 공유 폴더 링크](https://drive.google.com/drive/folders/1C_vmckfBTH4Axhf9R7ONxKg1lEjgyFAv?usp=sharing)를 클릭하세요 (images 폴더가 열립니다)
2. 상단 좌측의 폴더 이름 **images** 옆의 ▾ (드롭다운)를 클릭하세요
3. **정리** → **바로가기 추가** → **내 드라이브** 선택
4. 아래 셀을 실행하세요 (팝업이 뜨면 인증을 완료해주세요)

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
images = "/content/drive/MyDrive/images"
print(f"이미지 폴더: {images}")
print(f"이미지 수: {len([f for f in os.listdir(images) if f.endswith('.png')])}장")

In [ ]:
# reference 번역 데이터 다운로드 (평가 단계에서 사용)
!gdown 1AdmJeR14otr2L312F2JErWbIg1rVyRI3 -O references.csv -q
print("references.csv 다운로드 완료!")

### 0.2 패키지 설치

필요한 라이브러리를 설치합니다. 1~2분 정도 걸립니다.
dependency 관련 빨간 글씨(ERROR)가 뜰 수 있는데, 무시해도 됩니다.

설치가 끝나면 **반드시 런타임을 재시작**해주세요:
- 상단 메뉴 → **런타임** → **세션 다시 시작**
- 재시작 후 **이 셀은 건너뛰고** 다음 셀(0.3)부터 실행하세요

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*HF_TOKEN.*")

!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q accelerate sacrebleu pandas

### 0.3 GPU 메모리 확인 함수

현재 GPU 메모리 사용량을 확인하는 함수입니다. 워크샵 중 언제든 `gpu_mem()` 을 실행하면 확인할 수 있습니다.

In [ ]:
import torch, psutil, os

def gpu_mem():
    # RAM
    ram = psutil.virtual_memory()
    print(f"RAM: {ram.used / 1024**3:.1f}GB / {ram.total / 1024**3:.1f}GB ({ram.percent}%)")
    # VRAM
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            used = torch.cuda.memory_allocated(i) / 1024**3
            total = torch.cuda.get_device_properties(i).total_memory / 1024**3
            reserved = torch.cuda.memory_reserved(i) / 1024**3
            print(f"GPU {i} ({torch.cuda.get_device_name(i)}): {used:.1f}GB used / {total:.1f}GB total (reserved: {reserved:.1f}GB)")
    else:
        print("GPU not available")

gpu_mem()

## 1. 이미지에서 텍스트 추출하기 (OCR)

OCR(Optical Character Recognition)은 이미지 속의 글자를 컴퓨터가 읽을 수 있는 텍스트로 변환하는 기술입니다.

이 섹션에서는:
1. 이미지 파일을 확인하고
2. AI OCR 모델(GLM-OCR)로 텍스트를 추출한 뒤
3. CSV 파일로 저장합니다

### 1.1 이미지 샘플 확인하기 [바이브 코딩]

AI 챗봇에게 아래 프롬프트를 보내고, 생성된 코드를 다음 셀에 붙여넣어 실행해보세요.

**추천 프롬프트:**

> 다음 조건으로 Python 코드를 작성해줘:
> - `/content/drive/MyDrive/images` 폴더에 PNG 이미지 파일들이 있어
> - 그 중 처음 3개 이미지를 matplotlib으로 시각화해줘
> - 1행 3열로 배치하고, 파일명을 제목으로 표시해줘

In [ ]:
# 여기에 AI가 생성한 코드를 붙여넣으세요


<details>
<summary>코드 힌트 (위 셀이 잘 안 되면 펼쳐보세요)</summary>

```python
import matplotlib.pyplot as plt
from PIL import Image

png_files = sorted(f for f in os.listdir(images) if f.endswith(".png"))[:3]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, fname in zip(axes, png_files):
    img = Image.open(os.path.join(images, fname))
    ax.imshow(img)
    ax.set_title(fname, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()
```

</details>

### 1.2 OCR 모델 불러오기

GLM-OCR은 이미지 속 텍스트를 인식하는 AI 모델입니다 (0.9B 파라미터, 약 1.8GB).

아래 셀을 실행하면 모델이 다운로드됩니다. 처음 실행 시 1~2분 걸립니다.

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

ocr_model_name = "zai-org/GLM-OCR"
ocr_processor = AutoProcessor.from_pretrained(ocr_model_name)
ocr_model = AutoModelForImageTextToText.from_pretrained(
    ocr_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
print("GLM-OCR 모델 로드 완료!")
gpu_mem()

### 1.3 OCR 수행 함수 + 테스트

아래 셀을 실행하면 OCR 함수가 정의되고, 첫 번째 이미지로 테스트합니다.

In [ ]:
from PIL import Image

def ocr_single(image_path):
    """이미지 한 장에서 텍스트를 추출합니다."""
    msgs = [
        {
            "role": "user",
            "content": [
                {"type": "image", "url": image_path},
                {"type": "text", "text": "Text Recognition:"},
            ],
        }
    ]
    inp = ocr_processor.apply_chat_template(
        msgs, tokenize=True, return_tensors="pt",
        return_dict=True, add_generation_prompt=True,
    ).to(ocr_model.device)

    with torch.inference_mode():
        out = ocr_model.generate(**inp, max_new_tokens=2048)

    text = ocr_processor.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)
    return text

# 테스트: 첫 번째 이미지
test_files = sorted(f for f in os.listdir(images) if f.endswith(".png"))
result = ocr_single(os.path.join(images, test_files[0]))
print(f"=== {test_files[0]} ===")
print(result)

### 1.4 전체 이미지 OCR + CSV 저장 [바이브 코딩]

AI 챗봇에게 아래 정보를 전달하고, 코드를 요청해보세요.

**AI에게 전달할 정보 (Context):**

> 다음 조건으로 Python 코드를 작성해줘:
> - `images` 변수에 이미지 폴더 경로가 들어있어
> - `ocr_single(image_path)` 함수가 이미지 경로를 받아 텍스트(문자열)를 반환해
> - 이미지 파일은 `.png` 형식이고, `sorted()`로 정렬해서 처리해줘
> - 처음 10장만 처리해줘
> - 결과를 pandas DataFrame으로 만들고 `source.csv`로 저장해줘
> - CSV 열: `page_id` (번호), `filename` (파일명), `src` (OCR 추출 텍스트)
> - 진행 상황을 출력해줘

In [ ]:
# 여기에 AI가 생성한 코드를 붙여넣으세요


<details>
<summary>코드 힌트 (위 셀이 잘 안 되면 펼쳐보세요)</summary>

```python
import pandas as pd

png_files = sorted(f for f in os.listdir(images) if f.endswith(".png"))[:10]
results = []

for i, fname in enumerate(png_files):
    print(f"{i+1}/{len(png_files)} 처리 중: {fname}")
    text = ocr_single(os.path.join(images, fname))
    results.append({"page_id": i+1, "filename": fname, "src": text})

df = pd.DataFrame(results)
df.to_csv("source.csv", index=False)
print(f"\n저장 완료! {len(df)}페이지 -> source.csv")
df.head()
```

</details>

**파일 확인:** 좌측 **폴더 아이콘**을 클릭하여 `source.csv`가 생성되었는지 확인하세요. 더블클릭하면 내용을 미리볼 수 있습니다.

## 2. LLM을 이용하여 번역하기

LLM(Large Language Model)은 텍스트를 이해하고 생성하는 AI 모델입니다.
이 섹션에서는 LLM에게 번역을 시키되, **어떻게 번역할지를 지시하는 프롬프트를 직접 설계**해봅니다.

### 2.1 LLM 모델 + 생성 함수 불러오기

Qwen3.5-0.8B 모델을 사용합니다. 아래 두 셀을 순서대로 실행해주세요.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

llm_model_name = "Qwen/Qwen3.5-0.8B"
tokenizer = AutoTokenizer.from_pretrained(llm_model_name)
llm_model = AutoModelForCausalLM.from_pretrained(
    llm_model_name, torch_dtype=torch.bfloat16, device_map="auto"
)
print("LLM 모델 로드 완료!")
gpu_mem()

In [ ]:
def generate(messages, max_new_tokens=2048, temperature=0.7):
    """LLM에게 메시지를 보내고 응답을 받는 함수"""
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer(text, return_tensors="pt").to(llm_model.device)
    input_length = inputs["input_ids"].shape[1]

    with torch.inference_mode():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=True,
        )

    result = tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)
    return result

# 테스트
test = generate([{"role": "user", "content": "Say hello in Korean"}])
print(f"테스트 응답: {test}")

### 2.2 번역 프롬프트 설계하기 [프롬프트 엔지니어링]

LLM에게 번역을 시킬 때, **어떻게 지시하느냐(프롬프트)**에 따라 결과가 크게 달라집니다.

**이번 워크샵의 원문 특성:**
- 경제학 학술 서적 (F.A. Hayek 관련 저서)
- 격식체 영어, 긴 문장, 복잡한 구문
- 학술 용어 및 고유명사 포함 (예: Professor O'Driscoll, socialist economy)

아래 셀에서 두 가지 프롬프트를 직접 작성해보세요.

**프롬프트에 반영할 수 있는 사항:**
- **번역 대상 언어**: 본인의 전공 언어 또는 자신있는 언어
- **역할 부여**: "경제학 전문 번역가", "학술 서적 번역가" 등
- **어조(tone)**: 학술 번역에 적합한 격식체
- **고유명사 처리**: 인명은 음차 표기? 원문 유지?
- **문장 길이**: 긴 문장을 자연스럽게 나눌 것인지, 원문 구조를 유지할 것인지

**프롬프트 작성이 어렵다면?** AI 챗봇에게 프롬프트 자체를 만들어달라고 요청해보세요:

> 영어 경제학 학술 서적을 한국어로 번역하려고 해. LLM에게 보낼 system prompt와 user prompt를 작성해줘. 격식체로 번역하되, 인명은 원문 그대로 유지하도록 지시해줘.

셀을 실행하면 테스트 문장으로 결과를 확인할 수 있습니다.
**마음에 안 들면 이렇게 수정해보세요:**
- 번역 결과가 너무 딱딱하면 → 어조를 부드럽게 수정
- 용어가 일관되지 않으면 → glossary(용어집) 추가
- 의역이 너무 심하면 → "원문에 충실하게 번역하라" 지시 추가

In [ ]:
# 아래 두 프롬프트를 직접 작성해보세요.

system_prompt = ""  # 예: "You are a professional translator specializing in economics. Translate to Korean."

user_prompt_template = ""  # 예: "Translate the following English text to Korean:\n{text}"

# --- 아래는 수정하지 마세요 ---

def translate(text):
    """위에서 설계한 프롬프트로 텍스트를 번역합니다."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_template.format(text=text)},
    ]
    return generate(messages)

# 테스트: 프롬프트를 수정하고 이 셀을 다시 실행해보세요
result = translate("But the proof of the pudding is in the eating.")
print(f"원문: But the proof of the pudding is in the eating.")
print(f"번역: {result}")

### 2.3 대량 번역 + CSV 저장 [바이브 코딩]

위에서 만든 `translate()` 함수로 모든 페이지를 번역하는 코드를 AI에게 요청해보세요.

**AI에게 전달할 정보 (Context):**

> 다음 조건으로 Python 코드를 작성해줘:
> - `source.csv` 파일에 `page_id`, `filename`, `src` 열이 있어
> - `translate(text)` 함수가 영어 텍스트를 받아서 번역 결과(문자열)를 반환해
> - 모든 페이지를 번역해줘
> - 결과를 `output.csv`로 저장해줘 (열: `page_id`, `filename`, `src`, `tgt`)
> - 진행 상황과 각 페이지의 번역 결과 앞부분을 실시간으로 출력해줘

In [ ]:
# 여기에 AI가 생성한 코드를 붙여넣으세요


<details>
<summary>코드 힌트 (위 셀이 잘 안 되면 펼쳐보세요)</summary>

```python
import pandas as pd

df = pd.read_csv("source.csv")
results = []

for idx, row in df.iterrows():
    translation = translate(row["src"])
    results.append(translation)
    print(f"[{idx+1}/{len(df)}] {row['filename']} -> {translation[:60]}...")

df["tgt"] = results
df.to_csv("output.csv", index=False)
print(f"\n저장 완료! -> output.csv")
df.head()
```

</details>

**파일 확인 방법:** 셀 실행이 끝나면, 좌측 **폴더 아이콘**을 클릭하여 `output.csv` 파일이 생성되었는지 확인하세요. `source.csv`와 `output.csv` 두 파일이 모두 있어야 다음 단계로 진행할 수 있습니다.

## 3. 번역 품질 평가하기

번역의 품질을 어떻게 자동으로 측정할 수 있을까요? 두 가지 방식을 비교해봅니다:

| | BLEU | COMETKiwi |
|---|---|---|
| **방식** | 정답 번역(reference)과 단어 겹침 비교 | 원문 + 번역문만으로 품질 추정 |
| **정답 필요** | 필요함 (사람이 번역해야) | 필요 없음 |
| **장점** | 직관적, 빠름 | 대량 평가 가능, 의역에 강함 |
| **단점** | 같은 의미인데 다른 단어 → 낮은 점수 | 모델 의존적 |

### 3.1 BLEU (Reference-Based)

BLEU는 기계번역의 가장 대표적인 평가 지표입니다.
교수가 미리 준비한 reference 번역과 비교하여 점수를 계산합니다.

In [ ]:
import pandas as pd
from sacrebleu.metrics import BLEU

df_output = pd.read_csv("output.csv")

# 교수가 준비한 reference 번역 다운로드
!gdown 1AdmJeR14otr2L312F2JErWbIg1rVyRI3 -O references.csv -q
df_ref = pd.read_csv("references.csv")

bleu = BLEU(effective_order=True)
n_eval = min(len(df_ref), len(df_output))

print(f"== BLEU Score (높을수록 좋음) ==")
print(f"== {n_eval}페이지 평가 ==\n")
bleu_scores = []
for i in range(n_eval):
    hyp = str(df_output.iloc[i]["tgt"])
    ref = str(df_ref.iloc[i]["ref"])
    score = bleu.sentence_score(hyp, [ref])
    bleu_scores.append(score.score)
    print(f"page {i+1} BLEU: {score.score:.1f}")
    print(f"  번역: {hyp[:80]}...")
    print(f"  정답: {ref[:80]}...")
    print()

print(f"평균 BLEU: {sum(bleu_scores)/len(bleu_scores):.1f}")

### 3.2 평가 모델 설치 + Hugging Face 토큰 설정

COMETKiwi 모델은 Hugging Face에서 이용약관 동의가 필요합니다. 아래 순서를 따라주세요.

**Step 1: Hugging Face 가입 + 토큰 발급**
1. [huggingface.co](https://huggingface.co)에 가입 또는 로그인
2. 우측 상단 프로필 → **Settings** → **Access Tokens**
3. **Create new token** → 이름 입력 → **Read** 권한 → 생성
4. 생성된 토큰(`hf_...`)을 복사

**Step 2: 모델 이용약관 동의**
1. [COMETKiwi 모델 페이지](https://huggingface.co/Unbabel/wmt22-cometkiwi-da)에 접속
2. 페이지 상단의 이용약관에 **동의(Agree)**

**Step 3: Colab 보안 비밀에 토큰 등록**
1. Colab 좌측 사이드바에서 **열쇠 아이콘** (보안 비밀)을 클릭
2. **+ 새 보안 비밀 추가** 클릭
3. 이름: `HF_TOKEN` / 값: 복사한 토큰 붙여넣기
4. **노트북 액세스** 토글을 켜기

In [ ]:
!pip install -q unbabel-comet

# Colab 보안 비밀에서 HF 토큰 불러오기
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
print("HF 토큰 설정 완료!")

### 3.3 COMETKiwi (Reference-Free Quality Estimation)

COMETKiwi는 정답 번역 없이 **원문과 번역문만으로** 품질을 추정합니다.
전체 번역 결과에 대해 점수를 매겨봅시다. 모델 다운로드에 1~2분 걸립니다.

In [ ]:
import pandas as pd
from comet import download_model, load_from_checkpoint

# 번역 결과 다시 불러오기 (런타임 재시작 후)
df_output = pd.read_csv("output.csv")

comet_model_path = download_model("Unbabel/wmt22-cometkiwi-da")
comet_model = load_from_checkpoint(comet_model_path)

data = [
    {"src": str(row["src"]), "mt": str(row["tgt"])}
    for _, row in df_output.iterrows()
]

output = comet_model.predict(data, batch_size=8)

df_output["cometkiwi"] = output.scores
print("== COMETKiwi Score (높을수록 좋음) ==\n")
print(f"전체 평균: {output.system_score:.4f}\n")

for _, row in df_output.iterrows():
    print(f"page {row['page_id']}: {row['cometkiwi']:.4f} | {row['src'][:40]}... -> {row['tgt'][:40]}...")

# COMETKiwi 점수를 output.csv에 저장
df_output.to_csv("output.csv", index=False)
print("\noutput.csv에 cometkiwi 점수 추가 저장 완료!")

### 3.4 두 지표 비교 + 결과 정리 [바이브 코딩]

BLEU와 COMETKiwi 점수를 한눈에 비교할 수 있는 `comparisons.csv`를 만들어봅시다.

**AI에게 전달할 정보 (Context):**

> 다음 조건으로 Python 코드를 작성해줘:
> - `output.csv`에 `page_id`, `filename`, `src`, `tgt`, `cometkiwi` 열이 있어
> - `references.csv`에 `page_id`, `filename`, `ref` 열이 있어
> - 두 파일을 `page_id` 기준으로 합쳐서 `comparisons.csv`를 만들어줘
> - 각 페이지에 대해 BLEU 점수도 계산해서 `bleu` 열로 추가해줘 (sacrebleu 사용)
> - 최종 열: `page_id`, `src`, `tgt`, `ref`, `bleu`, `cometkiwi`
> - 마지막에 전체 결과를 출력하고, BLEU와 COMETKiwi 점수 차이가 큰 페이지 상위 3개를 보여줘

In [ ]:
# 여기에 AI가 생성한 코드를 붙여넣으세요


<details>
<summary>코드 힌트 (위 셀이 잘 안 되면 펼쳐보세요)</summary>

```python
import pandas as pd
from sacrebleu.metrics import BLEU

df_output = pd.read_csv("output.csv")
df_ref = pd.read_csv("references.csv")

df = df_output.merge(df_ref[["page_id", "ref"]], on="page_id", how="inner")

bleu = BLEU(effective_order=True)
df["bleu"] = df.apply(lambda r: bleu.sentence_score(str(r["tgt"]), [str(r["ref"])]).score, axis=1)

df = df[["page_id", "src", "tgt", "ref", "bleu", "cometkiwi"]]
df.to_csv("comparisons.csv", index=False)

print(f"comparisons.csv 저장 완료! ({len(df)}페이지)\n")
print(df[["page_id", "bleu", "cometkiwi"]].to_string(index=False))

print("\n== BLEU vs COMETKiwi 차이가 큰 페이지 ==")
df["gap"] = abs(df["bleu"] / 100 - df["cometkiwi"])
for _, row in df.nlargest(3, "gap").iterrows():
    print(f"\npage {row['page_id']}: BLEU={row['bleu']:.1f}, COMETKiwi={row['cometkiwi']:.4f}")
    print(f"  번역: {row['tgt'][:80]}...")
```

</details>

**파일 확인:** 좌측 폴더 아이콘에서 `comparisons.csv`를 확인하세요.

**생각해볼 질문:**
- BLEU는 낮은데 COMETKiwi는 높은 페이지가 있나요? 왜 그럴까요?
- 번역 전문가의 관점에서, 어떤 지표가 더 신뢰할 만한가요?
- 자동 평가만으로 번역 품질을 판단할 수 있을까요?

## 4. 제출

이클래스에 아래 파일 3가지와 소감을 제출해주세요.

**파일:**
- `source.csv` — OCR로 추출한 영어 원문 (페이지 단위)
- `output.csv` — LLM이 번역한 결과 (COMETKiwi 점수 포함)
- `comparisons.csv` — BLEU + COMETKiwi 비교 결과
- 현재 작업한 `.ipynb` 파일 (파일명을 `midterm_workshop_학번.ipynb`로 변경하여 저장)

**소감 (자유 서술):**
1. LLM(Qwen3.5-0.8B)이 수행한 번역 결과에 대한 번역 전문가로서의 생각
2. 프롬프트를 바꿨을 때 번역 결과가 어떻게 달라졌나요?
3. BLEU와 COMETKiwi 점수가 서로 다른 페이지가 있었나요? 왜 그럴까요?
4. AI를 활용한 번역 워크플로우에 대한 의견

---

수고하셨습니다.